# 04 - Tournament Simulation

Monte Carlo simulate the full 48-team 2026 World Cup and visualize title and advancement probabilities.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from wcpredictor.config import default_config
from wcpredictor.data import load_matches, load_teams, load_groups
from wcpredictor.data.preprocess import build_training_matches, team_match_counts

config = default_config()
plt.rcParams["figure.figsize"] = (9, 4.5)

from wcpredictor.models import PoissonModel
from wcpredictor.simulation import TournamentSimulator
import numpy as np

In [ ]:
tr = build_training_matches(load_matches(config), config)
poisson = PoissonModel(config).fit(tr)
groups = load_groups(config)
sim = TournamentSimulator(poisson, groups, config)
report = sim.run(n_simulations=3000)
report.table.head(15)

## Title probabilities (top 15)

In [ ]:
top = report.table.head(15).iloc[::-1]
ax = top.plot.barh(x='team', y='p_winner', legend=False, color='#C44E52')
ax.set_title('Probability of winning World Cup 2026')
ax.set_xlabel('P(win)'); plt.tight_layout(); plt.show()

## Stage-by-stage funnel for the favourites

In [ ]:
stages = ['p_advance', 'p_round_of_16', 'p_quarterfinal',
          'p_semifinal', 'p_final', 'p_winner']
labels = ['R32', 'R16', 'QF', 'SF', 'Final', 'Win']
fig, ax = plt.subplots()
for _, row in report.table.head(5).iterrows():
    ax.plot(labels, [row[s] for s in stages], marker='o', label=row['team'])
ax.set_ylabel('probability'); ax.set_title('Run to the title')
ax.legend(); plt.tight_layout(); plt.show()

## One simulated tournament (example bracket)

In [ ]:
rng = np.random.default_rng(7)
result = sim.simulate_once(rng)
print('Champion:', result['champion'])
print('Finalists:', sorted(result['reached']['final']))
print('Semifinalists:', sorted(result['reached']['semifinal']))

## Probability of topping the group

In [ ]:
# Re-run focusing on group winners would require per-group stats;
# here we show advancement probability grouped by draw group.
report.table.groupby('group')['p_advance'].mean().sort_values(ascending=False)